In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

#openCV görüntüleri BGR formatında kabul eder. Bizim alıştığımız görüntüler RGB formatındadır.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. ADIM: Görüntüyü OpenCV ile okuyoruz (Varsayılan: BGR)
img = cv2.imread(r"ornek_goruntu.png")

# --- HATALI GÖSTERİM (Matplotlib BGR Beklemez!) ---
# Bu satırı çalıştırdığında renklerin birbirine karıştığını göreceksin.
plt.imshow(img) 
plt.title("Hatalı Renkler (BGR formatı RGB sanıldı)")
plt.show()

# --- DOĞRU GÖSTERİM (Dönüşüm Şart) ---
# Matplotlib'in dili olan RGB'ye geçiş yapıyoruz.
img_rgb = cv2.cvtColor(
    src=img, 
    code=cv2.COLOR_BGR2RGB # BGR -> RGB köprüsü 
)

plt.imshow(img_rgb)
plt.title("Doğru Renkler (RGB formatına dönüştürüldü)")
plt.show()

OpenCV'de HSV değerleri şu aralıklarda tanımlanır:

    H (Hue - Renk Özü): 0 - 179 

    S (Saturation - Doygunluk): 0 - 255 

    V (Value - Parlaklık): 0 - 255 

1. Neden H (0 - 180) Aralığında?

Beyazın kendine has bir rengi yoktur. Bir nesne beyazsa, hangi renk özünde (kırmızı, mavi, yeşil) olduğunun bir önemi kalmaz. Bu yüzden lower kısmına 0, upper kısmına 180 vererek tüm renk spektrumunu "beyaz olabilir" diyerek kapsıyoruz.

2. Neden S (0 - 50) Aralığında?

Doygunluk (S), rengin ne kadar "saf" olduğunu söyler.

    0: Renksiz (Gri tonları/Beyaz) demektir.

    255: Tam doygun (Çok canlı renk) demektir.
    Beyaz şeritleri yakalamak için doygunluğun çok düşük olması gerekir. Eğer upper_white içindeki 50 değerini 100 yaparsan, yoldaki açık mavi veya soluk sarı pikselleri de "beyaz" sanıp maskeye dahil edersin.


3. Neden V (200 - 255) Aralığında?

Parlaklık (V), beyazı ayırt eden asıl anahtardır.

    255: Tam parlak (Işık vuran beyaz).

    200: Biraz daha mat, griye çalan beyaz.
    Alt sınırı 200 belirleyerek, asfaltın koyu gri kısımlarını eliyoruz ve sadece "parlayan" yol çizgilerine odaklanıyoruz.

In [ ]:
import cv2
import numpy as np

# img = cv2.imread("gta5_traffic.jpg") 

# --- 1. ADIM: HAZIRLIK (HSV FORMATINA DÖNÜŞÜM) ---
# inRange her zaman HSV formatında daha kararlı çalışır çünkü ışık değişimlerinden az etkilenir.
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# --- 2. ADIM: RENK SINIRLARINI BELİRLE ---
# HSV formatında (Hue, Saturation, Value) alt ve üst sınırları tanımlıyoruz.
lower_white = np.array([0, 0, 200])    # Alt Sınır: Çok az griye çalan kirli beyaz
upper_white = np.array([180, 50, 255])  # Üst Sınır: Tam parlak beyaz


# --- 3. ADIM: MASKEYİ OLUŞTUR ---
mask_white = cv2.inRange(
    src=hsv,           # Giriş görüntüsü (Genellikle HSV uzayında) 
    lowerb=lower_white, # Aralığın en düşük (alt) sınır değerleri 
    upperb=upper_white  # Aralığın en yüksek (üst) sınır değerleri 
)

# Sonuçta 'mask' değişkeni sadece sarı piksellerin beyaz olduğu siyah-beyaz bir resimdir.

plt.imshow(mask_white, cmap="gray")
plt.title("Beyaz Şerit Maskesi")
plt.show() #

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. ADIM: Trafik lambası ve fren ışıkları olan bir GTA 5 sahnesi yüklüyoruz
img = cv2.imread("ornek_goruntu_gece.jpg") # Orijinal BGR Görüntü

# --- 2. ADIM: KANALLARI AYIR (split) ---
# Dikkat: OpenCV BGR çalışır! Sıralama: b (0), g (1), r (2)
b, g, r = cv2.split(
    m=img # Renkli görüntüyü 3 ayrı gri matrise böler
)

# --- 3. ADIM: SADECE KIRMIZI KANALI GÖRSELLEŞTİR ---
# r değişkeni artık tek kanallı (gri) bir görüntüdür.
# Beyaz/Parlak yerler: Orijinal görüntüde Kırmızı'nın en yoğun olduğu yerler.
# Siyah/Koyu yerler: Orijinal görüntüde Kırmızı'nın olmadığı yerler (Örn: Mavi gökyüzü).

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Orijinal Gorus")
plt.subplot(1, 2, 2); plt.imshow(r, cmap="gray"); plt.title("Gri Tonlamali Kirmizi Kanal Filtresi") # cmap="gray" parametresi şarttır!
plt.show() 

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. ADIM: Trafik lambası olan bir GTA 5 sahnesi yüklüyoruz
img = cv2.imread("ornek_goruntu_gece.jpg") 

# --- 2. ADIM: KANALLARI AYIR (split) ---
# Unutma: OpenCV BGR çalışır! Sıralama b, g, r şeklindedir.
b, g, r = cv2.split(
    m=img # Renkli görüntüyü 3 ayrı gri matrise böler
)

# --- 3. ADIM: MANİPÜLASYON (Kırmızıyı İzole Et) ---
# Kırmızı ışığa odaklanmak için Mavi ve Yeşil kanalları sıfırlıyoruz.
# np.zeros_like, verdiğimiz matrisle aynı boyutta ama içi 0 (siyah) bir matris oluşturur.
zeros = np.zeros_like(b) 

# --- 4. ADIM: KANALLARI BİRLEŞTİR (merge) ---
# Sadece kırmızı kanalın dolu olduğu yeni bir renkli görüntü oluşturuyoruz.
red_only_vision = cv2.merge(
    mv=[zeros, zeros, r] # [Mavi=0, Yesil=0, Kirmizi=r]
)

# --- GÖRSELLEŞTİRME ---
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Orijinal Gorus")
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(red_only_vision, cv2.COLOR_BGR2RGB)); plt.title("Kirmizi Isik Odakli Gorus")
plt.show()